# 05 — HIP-4 binary fair value (P5)

**What this notebook answers**

- HIP-4 binary `BTC > 79980 by May 5 06:00 UTC` is split into two CLOBs (`#30` YES / `#31` NO). The two best-bids should sum to **≤ 1.0** and best-asks to **≥ 1.0** — otherwise free arb.
- Theoretical fair value via Black–Scholes digital: `P_yes = N(d2)` where
  `d2 = (ln(S/K) − σ²(T−t)/2) / (σ√(T−t))`.
- We use HL perp mid as `S`, the strike from market metadata as `K`, and a
  realised vol estimate as `σ`. Compare to traded prices and the implied vol you'd
  back out from market price.
- Cross-book arb spread: `1 − (bid_yes + bid_no)`.

In [ ]:
from hlanalysis.analysis import duck, glob_for, load_df, set_mpl_defaults
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from scipy.stats import norm
set_mpl_defaults()

# Pull market metadata for #30 and #31
mm = load_df(f'''
SELECT symbol, exchange_ts, keys, values
FROM read_parquet('{glob_for(venue='hyperliquid', product_type='prediction_binary', event='market_meta')}',
                  hive_partitioning=true)
ORDER BY exchange_ts DESC
''')
mm

In [ ]:
# Extract strike + expiry per symbol.
# HIP-4 metadata uses keys: outcome_idx, side_idx, side_name, outcome_name,
# class, underlying, expiry (e.g. '20260506-0600'), targetPrice, period.
def kv_lookup(row):
    d = dict(zip(row['keys'], row['values']))
    exp_raw = d.get('expiry','')
    try:
        exp_ts = pd.to_datetime(exp_raw, format='%Y%m%d-%H%M', utc=True) if exp_raw else None
    except Exception:
        exp_ts = None
    try:
        K = float(d.get('targetPrice','nan'))
    except Exception:
        K = float('nan')
    return pd.Series({
        'strike': K,
        'expiry': exp_ts,
        'side_name': d.get('side_name',''),
        'underlying': d.get('underlying',''),
        'period': d.get('period',''),
    })

latest_meta = mm.sort_values('exchange_ts').groupby('symbol').tail(1).reset_index(drop=True)
meta = pd.concat([latest_meta[['symbol']], latest_meta.apply(kv_lookup, axis=1)], axis=1)
meta

## 1. Cross-book arb spread

If `bid_yes + bid_no > 1.0` we could lift both bids and earn the difference at
settlement. If `ask_yes + ask_no < 1.0` we can buy both for less than 1. Either is
free money — should be ~0 net of fees.

In [ ]:
def bbo_grid(symbol, freq='5s'):
    g = glob_for(venue='hyperliquid', product_type='prediction_binary', event='bbo', symbol=symbol)
    df = load_df(f'''
        SELECT exchange_ts, bid_px, bid_sz, ask_px, ask_sz
        FROM read_parquet('{g}', hive_partitioning=true)
        WHERE bid_px>0 AND ask_px>0 ORDER BY exchange_ts
    ''')
    if df.empty: return df
    df['t'] = pd.to_datetime(df.exchange_ts, unit='ns', utc=True)
    return df.set_index('t').resample(freq).last().ffill().dropna()

yes = bbo_grid('#30').rename(columns=lambda c: c+'_yes')
no  = bbo_grid('#31').rename(columns=lambda c: c+'_no')
both = yes.join(no, how='inner').dropna()
both['bid_sum'] = both.bid_px_yes + both.bid_px_no
both['ask_sum'] = both.ask_px_yes + both.ask_px_no
both[['bid_sum','ask_sum']].describe()

In [ ]:
fig, ax = plt.subplots(figsize=(11,3.8))
ax.plot(both.index, both.bid_sum, label='bid_yes + bid_no')
ax.plot(both.index, both.ask_sum, label='ask_yes + ask_no')
ax.axhline(1.0, color='k', lw=0.7)
ax.set_ylabel('sum'); ax.set_title('HIP-4 cross-book sums (should bracket 1.0)')
ax.legend(); plt.tight_layout(); plt.show()

## 2. Theoretical fair value vs traded mid

We estimate σ from HL perp mid log-returns over the available window (annualized).
Then `P_yes = N(d2)`.

In [ ]:
# Volatility from HL perp BTC mid
hl = load_df(f'''
SELECT exchange_ts, (bid_px+ask_px)/2.0 AS mid
FROM read_parquet('{glob_for(venue='hyperliquid', product_type='perp', event='bbo', symbol='BTC')}',
                  hive_partitioning=true)
WHERE bid_px>0 AND ask_px>0 ORDER BY exchange_ts
''')
hl['t'] = pd.to_datetime(hl.exchange_ts, unit='ns', utc=True)
hl = hl.set_index('t')[['mid']].resample('1s').last().ffill().dropna()
ret = np.log(hl['mid']).diff().dropna()
# annualised vol from 1s log-rets
sigma_ann = ret.std() * np.sqrt(365*24*3600)
print(f'Realized 1s vol (annualised): {sigma_ann:.3%}')

In [ ]:
# Theoretical YES price using #30 metadata (strike + expiry)
m = meta[meta.symbol=='#30'].iloc[0] if (meta.symbol=='#30').any() else None
if m is None or pd.isna(m.strike) or m.expiry is None:
    print('No #30 metadata yet — print of meta:'); print(meta)
else:
    K = float(m.strike); T_expiry = pd.Timestamp(m.expiry).tz_convert('UTC')
    print(f'#30 strike={K:,.0f}  expiry={T_expiry}')

    grid = both.join(hl[['mid']].rename(columns={'mid':'S'}).resample('5s').last().ffill(), how='inner').dropna()
    tau = (T_expiry - grid.index).total_seconds() / (365*24*3600)
    tau = np.clip(tau, 1e-9, None)
    d2 = (np.log(grid['S']/K) - 0.5 * sigma_ann**2 * tau) / (sigma_ann * np.sqrt(tau))
    grid['theo_yes'] = norm.cdf(d2)
    grid['mid_yes']  = 0.5*(grid.bid_px_yes + grid.ask_px_yes)
    grid['mid_no']   = 0.5*(grid.bid_px_no  + grid.ask_px_no)

    fig, axes = plt.subplots(2,1, figsize=(11,6), sharex=True)
    axes[0].plot(grid.index, grid['mid_yes'], label='mid YES (#30)')
    axes[0].plot(grid.index, 1-grid['mid_no'], label='1 − mid NO (#31)')
    axes[0].plot(grid.index, grid['theo_yes'], label=f'N(d2) σ={sigma_ann:.0%}', ls='--')
    axes[0].set_ylabel('YES price'); axes[0].legend()
    axes[0].set_title(f'YES vs theoretical  (K={K:,.0f})')

    axes[1].plot(grid.index, grid['S'], color='C3'); axes[1].set_ylabel('BTC perp mid (S)')
    plt.tight_layout(); plt.show()

    grid[['S','mid_yes','mid_no','theo_yes']].tail()

## 3. Implied σ time series

Given traded YES mid, invert `N(d2)` for σ at each timestamp. A spike vs
realised σ is a cleaner signal than raw price for measuring market disagreement.

In [ ]:
def implied_vol(p_yes, S, K, tau):
    # find σ s.t. N(d2) = p_yes
    if not (0 < p_yes < 1) or tau <= 0 or S <= 0 or K <= 0: return np.nan
    target = norm.ppf(p_yes)  # = d2
    # d2(σ) = (ln(S/K) - 0.5 σ² τ) / (σ √τ)
    # solve quadratic in σ:  -0.5 τ σ² - (target √τ) σ + ln(S/K) = 0
    a = -0.5*tau; b = -target*np.sqrt(tau); c = np.log(S/K)
    disc = b*b - 4*a*c
    if disc < 0: return np.nan
    s1 = (-b + np.sqrt(disc))/(2*a); s2 = (-b - np.sqrt(disc))/(2*a)
    cands = [s for s in (s1, s2) if s and s > 0]
    return min(cands) if cands else np.nan

if 'grid' in globals():
    iv = []
    for ts, row in grid.iterrows():
        tau = (T_expiry - ts).total_seconds()/(365*24*3600)
        iv.append(implied_vol(row.mid_yes, row.S, K, tau))
    grid['impl_vol'] = iv
    fig, ax = plt.subplots(figsize=(11,3.6))
    ax.plot(grid.index, grid['impl_vol'], label='Implied σ from YES mid')
    ax.axhline(sigma_ann, color='k', ls='--', label=f'Realized σ = {sigma_ann:.0%}')
    ax.set_ylabel('annualised σ'); ax.legend()
    ax.set_title('HIP-4 implied vs realised vol')
    plt.tight_layout(); plt.show()

## Strategy hypotheses

- **Supports:** H?: (fill in after reviewing the chart above)
- **Rules out:** H?: (fill in if evidence rules it out)
- **Next probe:** (what would refine this further)